# Task 4: General Health Query Chatbot (Prompt Engineering)

### Objective
Create a friendly, safe chatbot that answers **general health-related questions** using a Large Language Model (LLM) via prompt engineering.

### Goal
* Provide helpful, easy-to-understand responses about general health topics.
* Strictly avoid giving medical diagnoses, prescriptions, or personalized medical advice.
* Include strong safety disclaimers and emergency detection.
* Demonstrate good prompt engineering and responsible AI practices.

**Important Note:** This chatbot is for educational/general information purposes only and is **not a substitute for professional medical advice.**

### Safety Architecture & Prompt Design
To fulfill the core skills required for this task, our safety framework relies on two distinct layers:

1. **The Python Layer (Hard Guardrails):** Scans user input *before* it hits the API. If emergency keywords are found (e.g., "chest pain", "stroke"), it short-circuits and demands the user seek emergency care.
2. **The Prompt Engineering Layer (Soft Guardrails):** A strictly defined system persona that enforces a friendly tone, demands an automatic disclaimer, and bans prescribing drug dosages.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load the local .env file automatically
load_dotenv()

# Grab the variable securely
hf_token = os.getenv("HF_TOKEN")

if not hf_token:
    raise ValueError("❌ System Error: HF_TOKEN is missing from your local configuration!")

# Initialize your client safely
client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=hf_token,
)
print("Connected securely! No keys are visible in the raw code cells. 🎉")

Connected securely! No keys are visible in the raw code cells. 🎉


### Chatbot Implementation
The function below (`health_query_bot`) implements both the hard and soft guardrails discussed above. It checks for severe symptoms first, and if none are found, passes the query to Llama-3.1 with our strict medical assistant system prompt.

In [2]:
def health_query_bot(prompt):
    """
    A prompt-engineered chatbot for general health queries featuring 
    both hard Python guardrails and soft LLM prompt guardrails.
    """
    
    # ---------------------------------------------------------
    # Layer 1: Python Hard Guardrails (Emergency Scanner)
    # ---------------------------------------------------------
    prompt_lower = prompt.lower()
    
    # UPDATED: Added more variations of the phrase to catch the test prompt!
    emergency_keywords = [
        "chest pain", "pain in my chest", "stroke", 
        "heart attack", "bleeding out", "can't breathe", 
        "suicide", "emergency"
    ]
    
    if any(keyword in prompt_lower for keyword in emergency_keywords):
        # The UI cell specifically looks for the word "EMERGENCY" to trigger the orange color
        return "⚠️ EMERGENCY ALERT: Your query contains severe symptoms. Please call emergency services (like 911) or go to the nearest hospital immediately. I am an AI, not a doctor, and cannot assist with medical emergencies."

    # ---------------------------------------------------------
    # Layer 2: Prompt Engineering Soft Guardrails
    # ---------------------------------------------------------
    system_prompt = (
        "You are a helpful, professional, and empathetic Health Assistant AI. "
        "Your goal is to provide general wellness information and health education. "
        "CRITICAL RULES: "
        "1. Never provide a formal medical diagnosis or prescribe medication. "
        "2. If the user describes emergency symptoms, urge them to seek immediate professional medical attention. "
        "3. Always recommend consulting a doctor for specific health concerns. "
        "4. Maintain a supportive, objective, and calm tone."
    )
    
    try:
        completion = client.chat.completions.create(
            model="meta-llama/Llama-3.1-8B-Instruct:novita",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7
        )
        return completion.choices[0].message.content
        
    except Exception as e:
        return f"API Error: Could not connect to the model. Details: {str(e)}"

### Model Evaluation and Testing
We will now test the chatbot using three distinct scenarios to ensure our safety layers are functioning correctly:
1. A general wellness query.
2. A simulated medical emergency (to trigger the Hard Guardrails).
3. A general lifestyle/diet query.

In [3]:
# Import the display tools from IPython
from IPython.display import display, Markdown

def render_chat_ui(query, response):
    """
    A minimalist, dark-themed UI renderer with distinct background panels 
    for the user query and system response.
    """
    is_emergency = "EMERGENCY" in response
    
    # Sleek visual styling: Orange + Siren for alerts, Cyan + Robot for standard
    accent_color = "#FF5C00" if is_emergency else "#00F0FF" 
    icon = "🚨" if is_emergency else "🤖"
    
    ui_template = f"""
<div style="background-color: #1E212B; padding: 12px 20px; border-radius: 6px; border-left: 4px solid {accent_color}; margin-bottom: 8px;">
<span style="color: {accent_color}; font-weight: bold; font-size: 0.85em; text-transform: uppercase; letter-spacing: 1.5px;">👤 User Input</span><br>
<span style="font-size: 1.1em; color: #FFFFFF;">{query}</span>
</div>

<div style="background-color: #0B0D12; padding: 16px 20px; border-radius: 6px; border: 1px solid #2D3748; margin-bottom: 30px;">
<span style="color: #9CA3AF; font-weight: bold; font-size: 0.85em; text-transform: uppercase; letter-spacing: 1.5px;">{icon} System Output</span>

{response}

</div>
"""
    # Display the compiled Markdown
    display(Markdown(ui_template))

# --- Testing the Chatbot ---
test_scenarios = [
    "What are some natural ways to reduce stress?",
    "I'm feeling a sharp pain in my chest, what should I do?",
    "How can I maintain a healthy diet?"
]

print("Initializing Chatbot UI...\n")

for query in test_scenarios:
    response = health_query_bot(query)
    render_chat_ui(query, response)

Initializing Chatbot UI...




<div style="background-color: #1E212B; padding: 12px 20px; border-radius: 6px; border-left: 4px solid #00F0FF; margin-bottom: 8px;">
<span style="color: #00F0FF; font-weight: bold; font-size: 0.85em; text-transform: uppercase; letter-spacing: 1.5px;">👤 User Input</span><br>
<span style="font-size: 1.1em; color: #FFFFFF;">What are some natural ways to reduce stress?</span>
</div>

<div style="background-color: #0B0D12; padding: 16px 20px; border-radius: 6px; border: 1px solid #2D3748; margin-bottom: 30px;">
<span style="color: #9CA3AF; font-weight: bold; font-size: 0.85em; text-transform: uppercase; letter-spacing: 1.5px;">🤖 System Output</span>

Reducing stress is essential for overall well-being, and there are many natural ways to achieve it. Here are some effective methods to consider:

1. **Mindfulness and Meditation**: Practice mindfulness meditation, deep breathing exercises, or yoga to calm your mind and body. You can use guided meditation apps like Headspace or Calm to get started.
2. **Exercise**: Engage in physical activities like walking, jogging, swimming, or cycling to release endorphins, also known as "feel-good" hormones. Exercise can help reduce stress and anxiety.
3. **Nature Therapy**: Spend time outdoors, and connect with nature by walking in parks, hiking, or simply sitting in a garden or on a balcony with plants. Being in nature can help reduce stress and improve mood.
4. **Herbal Remedies**: Explore herbal teas like chamomile, lavender, and peppermint, which are known for their calming effects. You can also try herbal supplements like ashwagandha or passionflower, but always consult with a healthcare professional before adding new supplements to your routine.
5. **Aromatherapy**: Inhale essential oils like lavender, bergamot, or frankincense, which can help calm your mind and body. Use a diffuser or apply a few drops to your pulse points.
6. **Sleep Hygiene**: Establish a consistent sleep schedule, avoid caffeine and electronics before bedtime, and create a relaxing sleep environment to improve the quality of your sleep.
7. **Social Connections**: Nurture your relationships with friends, family, or a therapist. Social support can help you feel less isolated and more supported, reducing stress and anxiety.
8. **Healthy Diet**: Focus on consuming a balanced diet rich in whole foods, fruits, vegetables, and lean proteins. Avoid sugary and processed foods that can exacerbate stress.
9. **Laughter and Play**: Incorporate activities that bring you joy, like reading, painting, or playing with pets. Laughter and play can help reduce stress and improve your mood.
10. **Time Management**: Prioritize tasks, set realistic goals, and take regular breaks to maintain a healthy work-life balance. This can help reduce stress and increase feelings of control.

Remember, everyone is unique, and what works for one person may not work for another. Experiment with different techniques to find what works best for you.

If you're experiencing overwhelming stress or anxiety, consider consulting a healthcare professional for personalized guidance and support.

How can I help you further?

</div>



<div style="background-color: #1E212B; padding: 12px 20px; border-radius: 6px; border-left: 4px solid #FF5C00; margin-bottom: 8px;">
<span style="color: #FF5C00; font-weight: bold; font-size: 0.85em; text-transform: uppercase; letter-spacing: 1.5px;">👤 User Input</span><br>
<span style="font-size: 1.1em; color: #FFFFFF;">I'm feeling a sharp pain in my chest, what should I do?</span>
</div>

<div style="background-color: #0B0D12; padding: 16px 20px; border-radius: 6px; border: 1px solid #2D3748; margin-bottom: 30px;">
<span style="color: #9CA3AF; font-weight: bold; font-size: 0.85em; text-transform: uppercase; letter-spacing: 1.5px;">🚨 System Output</span>

⚠️ EMERGENCY ALERT: Your query contains severe symptoms. Please call emergency services (like 911) or go to the nearest hospital immediately. I am an AI, not a doctor, and cannot assist with medical emergencies.

</div>



<div style="background-color: #1E212B; padding: 12px 20px; border-radius: 6px; border-left: 4px solid #00F0FF; margin-bottom: 8px;">
<span style="color: #00F0FF; font-weight: bold; font-size: 0.85em; text-transform: uppercase; letter-spacing: 1.5px;">👤 User Input</span><br>
<span style="font-size: 1.1em; color: #FFFFFF;">How can I maintain a healthy diet?</span>
</div>

<div style="background-color: #0B0D12; padding: 16px 20px; border-radius: 6px; border: 1px solid #2D3748; margin-bottom: 30px;">
<span style="color: #9CA3AF; font-weight: bold; font-size: 0.85em; text-transform: uppercase; letter-spacing: 1.5px;">🤖 System Output</span>

Maintaining a healthy diet is a crucial aspect of overall well-being. Here are some tips to help you make informed choices:

1. **Eat a variety of whole foods**: Focus on consuming a wide range of fruits, vegetables, whole grains, lean proteins, and healthy fats. These foods provide essential nutrients, fiber, and satiety.
2. **Hydrate adequately**: Drink plenty of water throughout the day, and limit sugary drinks and soda. Aim for at least eight cups (64 ounces) of water daily.
3. **Limit processed and packaged foods**: Try to avoid or limit foods that are high in added sugars, saturated fats, and sodium. These foods can be detrimental to your health in the long run.
4. **Incorporate healthy fats**: Nuts, seeds, avocados, and olive oil are rich in healthy fats that support heart health and provide energy.
5. **Be mindful of portion sizes**: Eat until you're satisfied, but avoid overeating. Use a food scale or measuring cups to gauge your portions.
6. **Cook at home**: Preparing meals at home allows you to control the ingredients and portion sizes. Aim to cook at home most nights of the week.
7. **Watch your sugar intake**: Limit added sugars, which can be found in foods like baked goods, candy, and sweetened beverages.
8. **Get enough fiber**: Aim for 25-30 grams of fiber per day from sources like fruits, vegetables, whole grains, and legumes.
9. **Stay informed**: Read food labels, and stay up-to-date on the latest nutrition research and recommendations.
10. **Consult a registered dietitian or healthcare professional**: If you have specific dietary needs or concerns, consult a registered dietitian or healthcare professional for personalized guidance.

Remember, developing healthy eating habits takes time and practice. Start by making small changes and gradually work your way towards a balanced diet.

Do you have any specific dietary concerns or preferences? I'm here to help you explore healthier options.

</div>


## Here User will be able to ask its own questions 

In [4]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Create a dedicated "Screen" for our chat history
chat_screen = widgets.Output()

# 2. Design the Custom Input Box
text_input = widgets.Text(
    value='',
    placeholder='Type your health question here... (e.g., How to reduce stress?)',
    description='👤 Query:',
    layout=widgets.Layout(width='75%', border='1px solid #00F0FF')
)

# 3. Design the Submit Button
submit_button = widgets.Button(
    description='Send Data 🚀',
    button_style='info', 
    layout=widgets.Layout(width='15%', border='1px solid #00F0FF')
)

# 4. The Logic: What happens when you press Send
def on_send(button_click_or_enter_key):
    # CRITICAL FIX: .strip() removes any invisible spaces the user might have typed!
    user_query = text_input.value.strip()
    
    # Ignore empty clicks
    if not user_query:
        return
        
    # Check for the kill command
    if user_query.lower() in ['exit', 'quit']:
        with chat_screen:
            # Send a styled shutdown message instead of standard print
            render_chat_ui(user_query, "System shutting down... Goodbye! 👋")
        
        # Lock the interface so no more messages can be sent
        text_input.disabled = True
        submit_button.disabled = True
        text_input.value = '[ SYSTEM OFFLINE ]'
        return

    # Clear the text box instantly for the next question
    text_input.value = ''
    
    # Route the output to our dedicated chat screen
    with chat_screen:
        response = health_query_bot(user_query)
        render_chat_ui(user_query, response)

# 5. Bind the Enter key and Button click to our logic
text_input.on_submit(on_send)
submit_button.on_click(on_send)

# 6. Group the input box and button together horizontally
input_dashboard = widgets.HBox([text_input, submit_button])

# --- Boot up the UI ---
print("Terminal Initialized. Type 'exit' to shut down.\n")

display(chat_screen)
display(input_dashboard)

Terminal Initialized. Type 'exit' to shut down.



C:\Users\M4 Tech\AppData\Local\Temp\ipykernel_15728\3474052323.py:52: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  text_input.on_submit(on_send)


Output()

### Explanation of Results and Final Insights

* **Test 1 (Stress Reduction):** The LLM successfully relied on the system prompt to deliver empathetic, general wellness advice without prescribing specific medical treatments.
* **Test 2 (Emergency Detection):** The Python Hard Guardrail successfully intercepted the phrase "pain in my chest". The API was bypassed entirely, saving compute resources and immediately delivering a critical safety warning.
* **Test 3 (Healthy Diet):** The model provided standard, objective health education while maintaining the professional tone requested in the system prompt.

**Conclusion:** By combining keyword-based Python filters with robust prompt engineering, we can create a highly effective and structurally safe conversational agent using open-source models like Llama-3.